# Examining MS SQL (TDS) traffic with Zeek

This is the hunt from [*Threat Hunting with Python Part 4: Examining Microsoft SQL Based Historian Traffic*](../docs/2018-03-threat-hunting-with-python-part-4-mssql-historian.md), redone with Zeek instead of tshark. The original notebook in this folder, `tds.ipynb`, drives tshark through pyshark and picks fields out of each packet; it still runs and is kept as written.

Here the same capture goes through Zeek with the [zeek-tds](https://github.com/dgunter/zeek-tds) analyzer, which understands the Tabular Data Stream protocol end to end: it reassembles messages across packets, resolves the numbered procedures drivers use for prepared statements, renders every parameter value by type, and writes logs that join to the rest of Zeek's output. [ParseZeekLogs](https://github.com/dgunter/ParseZeekLogs) then reads those logs into pandas. Zeek runs in Docker so nothing but Docker is needed on the machine; the same image is what you would run on a sensor.

In [1]:
import shutil
import subprocess
import tempfile
from pathlib import Path

import pandas as pd
from parsezeeklogs import ZeekLog

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

PCAP = Path("ms-sql-tds-rpc-requests.pcap").resolve()
IMAGE = "zeek-tds:0.2.0"

## Run Zeek over the capture

The image is Zeek 8.2 with the zeek-tds package installed; `zeek/Dockerfile` builds it in a few minutes if it is not already present.

In [2]:
def ensure_image():
    if subprocess.run(["docker", "image", "inspect", IMAGE], capture_output=True).returncode != 0:
        subprocess.run(
            ["docker", "build", "-q", "-t", IMAGE, str(Path("zeek").resolve())], check=True
        )


def run_zeek(pcap: Path) -> Path:
    out = Path(tempfile.mkdtemp(prefix="zeek-tds-"))
    subprocess.run(
        [
            "docker",
            "run",
            "--rm",
            "-v",
            f"{pcap.parent}:/pcap:ro",
            "-v",
            f"{out}:/data",
            "-u",
            "0",
            IMAGE,
            "-r",
            f"/pcap/{pcap.name}",
        ],
        check=True,
        capture_output=True,
    )
    return out


ensure_image()
logs = run_zeek(PCAP)
sorted(p.name for p in logs.glob("*.log"))

['conn.log',
 'packet_filter.log',
 'tds.log',
 'tds_rpc.log',
 'tds_sql_batch.log']

## Load the logs

Every Zeek log becomes a typed DataFrame. The `uid` column is the connection identifier that ties the TDS logs to `conn.log` and to each other.

In [3]:
def load(name: str) -> pd.DataFrame:
    with ZeekLog(logs / f"{name}.log") as log:
        df = pd.DataFrame(log)
    if "ts" in df:
        df["ts"] = pd.to_datetime(df["ts"], unit="s", utc=True)
    return df


conn = load("conn")
tds = load("tds")
batches = load("tds_sql_batch")
rpc = load("tds_rpc")
print(
    f"{len(conn)} connections, {len(tds)} TDS messages, "
    f"{len(batches)} SQL batches, {len(rpc)} procedure calls"
)
conn[
    [
        "ts",
        "uid",
        "id.orig_h",
        "id.orig_p",
        "id.resp_h",
        "id.resp_p",
        "service",
        "orig_bytes",
        "resp_bytes",
    ]
]

12 connections, 29 TDS messages, 3 SQL batches, 17 procedure calls


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,service,orig_bytes,resp_bytes
0,2009-04-28 00:18:37.888015032+00:00,CMV1YtcGQa28Aw3Li,10.111.111.111,1111,10.0.0.1,1433,None,482.0,392.0
1,2009-12-02 14:00:00.004436970+00:00,CyMGgL2gwVE3UoRR64,10.111.111.111,2222,10.0.0.1,1433,None,44.0,17.0
2,2009-12-02 14:00:00.022561073+00:00,CKrjw32unPYVCZru3h,10.111.111.111,3333,10.0.0.1,1433,None,NaN,NaN
3,2009-12-02 14:00:00.033700943+00:00,C2FSPlZyZ8faGM5lk,10.111.111.111,4444,10.0.0.1,1433,None,NaN,NaN
4,2009-12-02 14:00:00.716203928+00:00,CxgidU3ajiUBltwFG4,10.111.111.111,5555,10.0.0.1,1433,None,994.0,1143.0
5,2009-12-02 14:01:14.884130955+00:00,CElNO63RG0vn3FqZne,10.111.111.111,6666,10.0.0.1,1433,None,8339.0,0.0
6,2009-12-02 14:01:17.536189079+00:00,CIe7Xt2ea3S98hMYR6,10.111.111.111,7777,10.0.0.1,1433,None,NaN,NaN
7,2009-12-02 14:01:22.456089973+00:00,CAVjrH15Sb48LKvXMj,10.111.111.111,8888,10.0.0.1,1433,None,NaN,NaN
8,2010-07-02 11:00:44.666074991+00:00,CLajdH1f5ARDC3FFCc,10.111.111.111,33333,10.0.0.1,1433,None,NaN,NaN
9,2010-07-02 11:00:44.650715113+00:00,ChlDVK3XDEu7tWz8O7,10.111.111.111,22222,10.0.0.1,1433,None,NaN,NaN


## What kinds of messages, and which way

`tds.log` is the skeleton: one line per message after reassembly. The original notebook counted TDS packet types; here the counts are per message, and responses carry how many rows they returned.

In [4]:
tds["direction"] = tds["is_orig"].map({True: "client", False: "server"})
tds.groupby(["direction", "msg_type"]).agg(
    messages=("uid", "size"), bytes=("len", "sum"), packets=("packets", "sum"), rows=("rows", "sum")
)

messages  bytes  packets  rows
direction msg_type                                      
client    rpc                   16  12030       17   0.0
          sql_batch              3    400        3   0.0
server    tabular_result        10   1472       10   7.0

## What they typed: SQL batches

In [5]:
batches[["ts", "id.orig_p", "transaction_descriptor", "query"]]

,ts,id.orig_p,transaction_descriptor,query
0,2009-04-28 00:18:37.888015032+00:00,1111,0,set transaction isolation level read committed set implicit_transactions off
1,2009-12-02 14:00:00.004436970+00:00,2222,0,COMMIT TRANSACTION
2,2009-12-02 14:00:00.716203928+00:00,5555,0,set transaction isolation level read committed set implicit_transactions off


## What they called: procedures

This is where the 2018 analysis ran out of road. Drivers ship prepared statements as calls to `sp_prepexec` and `sp_execute`, addressed by number on the wire; the original notebook mapped the numbers to names by hand and never saw the SQL inside them. Zeek resolves the procedure, lifts the SQL text into `statement`, and keeps the handle that ties an `sp_execute` back to the `sp_prepexec` that prepared it.

In [6]:
rpc["procedure"].value_counts()

procedure
sp_prepexec                             5
sp_execute                              3
proc_GetMyExampleTableSampleMetaData    2
p_GetBogusData                          1
sp_executesql                           1
p_SaveExample                           1
p_SetBogusSample                        1
p_GetMyExampleTableRowCount             1
dbo.proc_GetMySampleDataItems           1
proc_FetchMyExampleData                 1
Name: count, dtype: int64

In [7]:
rpc[["ts", "id.orig_p", "procedure", "proc_id", "handle", "statement"]]

,ts,id.orig_p,procedure,proc_id,handle,statement
0,2009-04-28 00:18:37.918653011+00:00,1111,sp_prepexec,13.0,0,select * from test_table_1 where name = @P0 and id = @P1
1,2009-12-02 14:00:00.747371912+00:00,5555,sp_prepexec,13.0,0,"create table newsyb (column1 char(30) not null, column2 char(30) null,column3 char(30) null)"
2,2009-12-02 14:00:01.226156950+00:00,5555,sp_prepexec,13.0,0,"insert INTO newsyb (column1, column2, column3) VALUES ('first', 'second', 'third')"
3,2009-12-02 14:00:01.711920977+00:00,5555,sp_execute,12.0,2,NaN
4,2009-12-02 14:00:01.711920977+00:00,5555,sp_execute,12.0,2,NaN
5,2009-12-02 14:00:03.727473021+00:00,5555,sp_prepexec,13.0,0,select * from newsyb
6,2009-12-02 14:00:05.827626944+00:00,5555,sp_execute,12.0,3,NaN
7,2009-12-02 14:00:07.864718914+00:00,5555,sp_prepexec,13.0,0,drop table newsyb
8,2009-12-02 14:00:00.022561073+00:00,3333,p_GetBogusData,NaN,NaN,NaN
9,2009-12-02 14:00:00.033700943+00:00,4444,sp_executesql,NaN,NaN,"SELECT TOP 88 [dbo].[MyExampleTable].[ID], [dbo].[MyExampleTable].[EntityID], [dbo].[MyExampleTable].[EntityHistoryI..."


Every parameter is rendered by type, so the historian's own procedures show what they were asked for:

In [8]:
for _, row in rpc[~rpc["procedure"].str.startswith("sp_")].iterrows():
    print(f"{row['procedure']}")
    for p in row["parameters"]:
        print(f"    {p[:110]}")

p_GetBogusData
    @SearchType intn(1) = 1
    @MaxWaitTimeInSeconds intn(4) = 0
    @ProcessNegativeAck intn(1) = 0
p_SaveExample
    @LongParam nvarchar(max) = Studenckie Koło Przewodników Turystycznych w Gdańsku
zaprasza na:
XXXV Nocne Marsze
    @Operation intn(4) = 1
p_SetBogusSample
    @BogusDetailsID intn(8) = 74565
    @BogusStatusID intn(8) = 5
    @ResultCode nvarchar(1) = NULL
    @ResultMsg nvarchar(1) = NULL
    @ErrorCode nvarchar(1) = NULL
    @ErrorMsg nvarchar(1) = NULL
    @ExampleBogusGeneratedID nvarchar(1) = NULL
    @ExampleType intn(4) = 1
p_GetMyExampleTableRowCount
dbo.proc_GetMySampleDataItems
    @SampleItemId uniqueidentifier = 4EC31A66-A214-4853-A77E-E7060FFFFF07
    @DataId nvarchar(1) = NULL
    @DataItemType uniqueidentifier = BCB9459B-83A8-4564-B1D3-E9E198478F4E
    @TableId nvarchar(1) = NULL
    @MaxFetchSize intn(4) = 100
    @SomeOtherSampleId intn(4) = 0
proc_FetchMyExampleData
    @p1 uniqueidentifier = 67452301-AB89-EFCD-0123-456789ABCDEF
    @p

## The questions from the post

The post looked for two specific calls. Both are a filter away, and now the parameters come with them.

In [9]:
rpc.loc[
    rpc["procedure"] == "p_GetBogusData",
    ["ts", "id.orig_h", "id.resp_h", "procedure", "parameters"],
]

,ts,id.orig_h,id.resp_h,procedure,parameters
8,2009-12-02 14:00:00.022561073+00:00,10.111.111.111,10.0.0.1,p_GetBogusData,"[@SearchType intn(1) = 1, @MaxWaitTimeInSeconds intn(4) = 0, @ProcessNegativeAck intn(1) = 0]"


In [10]:
rpc.loc[
    rpc["procedure"].isin(["p_SetBogusSample", "sp_executesql"]),
    ["ts", "id.orig_h", "id.resp_h", "procedure", "statement", "parameters"],
]

,ts,id.orig_h,id.resp_h,procedure,statement,parameters
9,2009-12-02 14:00:00.033700943+00:00,10.111.111.111,10.0.0.1,sp_executesql,"SELECT TOP 88 [dbo].[MyExampleTable].[ID], [dbo].[MyExampleTable].[EntityID], [dbo].[MyExampleTable].[EntityHistoryI...","[@p1 nvarchar(467) = SELECT TOP 88 [dbo].[MyExampleTable].[ID], [dbo].[MyExampleTable].[EntityID], [dbo].[MyExampleT..."
11,2009-12-02 14:01:17.536189079+00:00,10.111.111.111,10.0.0.1,p_SetBogusSample,NaN,"[@BogusDetailsID intn(8) = 74565, @BogusStatusID intn(8) = 5, @ResultCode nvarchar(1) = NULL, @ResultMsg nvarchar(1)..."


## What tshark did not see

The original notebook's pyshark pass found 8 named procedures and 7 numbered ones. Zeek logs 17 procedure calls from the same 16 RPC messages. The two differences are structural, not a matter of field extraction:

- **Messages that span packets.** One RPC (`p_SaveExample`, with an 8 KB `nvarchar(max)` argument) arrives in two TDS packets. tshark dissects packet by packet and does not reassemble TDS messages, so it saw two RPC frames with no procedure in either. Zeek reassembles the message first.
- **Messages that hold several calls.** One 53-byte message carries two `sp_execute` calls separated by a batch flag. tshark reports the first procedure per packet; Zeek logs each call.

Both cases are visible in the Zeek output: RPC messages with more than one packet, and timestamps at which more than one procedure call was logged.

In [11]:
multi_packet = tds[(tds["msg_type"] == "rpc") & (tds["packets"] > 1)][
    ["ts", "id.orig_p", "len", "packets"]
]
multi_call = rpc.groupby("ts").size()
multi_call = multi_call[multi_call > 1]
print("RPC messages spanning several packets:")
print(multi_packet.to_string(index=False))
print("\nTimestamps with more than one procedure call in one message:")
print(
    rpc[rpc["ts"].isin(multi_call.index)][["ts", "id.orig_p", "procedure", "parameters"]].to_string(
        index=False
    )
)

RPC messages spanning several packets:
                                 ts  id.orig_p  len  packets
2009-12-02 14:01:14.884730101+00:00       6666 8323        2

Timestamps with more than one procedure call in one message:
                                 ts  id.orig_p  procedure        parameters
2009-12-02 14:00:01.711920977+00:00       5555 sp_execute [@p1 intn(4) = 2]
2009-12-02 14:00:01.711920977+00:00       5555 sp_execute [@p1 intn(4) = 2]


## Where this goes next

On a sensor the same analyzer runs continuously and adds a `tds_login.log` of who connected with which driver, a `tds_error.log` of failed logins and permission errors, and notices for brute force, dangerous procedures and unusual result sizes. The capture here predates TLS on the wire; on a modern network the analyzer hands encrypted sessions to Zeek's SSL analyzer and keeps the login negotiation.

In [12]:
shutil.rmtree(logs, ignore_errors=True)